# ReadBuddy Stage 3 — reproducible 64×64 Sinhala letter CNN
This notebook reproduces the **architecture** recovered from the shipped TFLite model, freezes a leakage-resistant split, trains a compact baseline on the same split, evaluates the held-out test set, exports TFLite, and checks prediction parity. It does not invent or pre-fill accuracy.


## 1. Free runtime setup
In Colab choose **Runtime → Change runtime type → T4 GPU** when available. Upload this project folder and the dataset, or mount Google Drive. Do not place Kaggle credentials inside a shared notebook.

In [ ]:
!python -m pip install -q -r /content/reading_assistant_app_updated/training/requirements-stage3.txt

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path('/content/reading_assistant_app_updated')
DATASET_DIR = Path('/content/sinhala_letter_454')  # edit this path
OUTPUT_DIR = PROJECT_ROOT / 'training/outputs/stage3'
MANIFEST = PROJECT_ROOT / 'training/splits/split_manifest.csv'
assert (PROJECT_ROOT / 'training/stage3_cnn_pipeline.py').is_file(), PROJECT_ROOT
assert DATASET_DIR.is_dir(), DATASET_DIR
print('Project:', PROJECT_ROOT)
print('Dataset:', DATASET_DIR)

## 2. Freeze the split before training
The command hashes every image, keeps byte-identical duplicates in one split, rejects identical bytes with conflicting class labels, and requires all numeric classes 1–454. Review and preserve the CSV and summary before continuing.

In [ ]:
import subprocess, sys
pipeline = PROJECT_ROOT / 'training/stage3_cnn_pipeline.py'
subprocess.run([sys.executable, str(pipeline), '--dataset', str(DATASET_DIR), '--output-dir', str(OUTPUT_DIR), '--manifest', str(MANIFEST), '--seed', '42', '--create-manifest', '--prepare-only'], check=True)

In [ ]:
import json
summary = json.loads(MANIFEST.with_suffix('.summary.json').read_text())
summary

## 3. Train, evaluate and export
This uses the already frozen manifest. Training-time choices are explicit: Adam, sparse categorical cross-entropy, early stopping on validation loss, seed 42, 64×64 grayscale input, [0,1] scaling, and no stochastic augmentation in this first faithful architecture reproduction.

In [ ]:
subprocess.run([sys.executable, str(pipeline), '--dataset', str(DATASET_DIR), '--output-dir', str(OUTPUT_DIR), '--manifest', str(MANIFEST), '--seed', '42', '--epochs', '40', '--batch-size', '32', '--patience', '6'], check=True)

## 4. Inspect retained evidence
Only values in these generated files should be copied into the thesis. Keep the full output folder with the submitted experiment record.

In [ ]:
comparison = json.loads((OUTPUT_DIR / 'model_comparison.json').read_text())
parity = json.loads((OUTPUT_DIR / 'keras_tflite_parity.json').read_text())
print(json.dumps(comparison, indent=2))
print(json.dumps(parity, indent=2))
print('Evidence files:')
for path in sorted(OUTPUT_DIR.iterdir()): print(' -', path.name)

## 5. Thesis interpretation
Report the custom CNN against the compact baseline on the **same held-out test samples**. Discuss macro-F1 and per-class recall alongside accuracy because 454-class imbalance can hide weak classes. Report calibration and Keras/TFLite parity. If only a subset was used, label it a pilot and never generalize it to all 454 classes.